In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

class BackdoorDataset(Dataset):
    def __init__(self, csv_path, img_root, split='train', processor=None):
        self.img_root = img_root
        self.processor = processor
        self.samples = []  # 用于存储展开后的数据

        print(f"正在加载数据集: {csv_path} | Split: {split}")
        df = pd.read_csv(csv_path)
        df = df[df['split'] == split].reset_index(drop=True)
        print(f"原始数据条数: {len(df)}")
        print(df.dtypes)
        print(df.head())

        # 预处理：展开多条 caption
        for idx, row in df.iterrows():
            img_path = os.path.join(self.img_root, row['split'], row['filename'])
            
            # 解析 raw 和 origin 列 (字符串列表 -> list)
            try:
                raw_captions = ast.literal_eval(row['raw'])
                origin_captions = ast.literal_eval(row['origin'])
            except:
                continue

            # 确保 raw 和 origin 数量一致
            if len(raw_captions) != len(origin_captions):
                continue

            # 将每一条 caption 拆分为独立样本
            for raw_cap, origin_cap in zip(raw_captions, origin_captions):
                self.samples.append({
                    'img_path': img_path,
                    'raw_caption': raw_cap,       # 可能包含触发词的文本
                    'origin_caption': origin_cap, # 原始干净文本
                    'poisoned': row['poisoned']
                })
        
        print(f"展开完成，共生成 {len(self.samples)} 个训练样本。")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        data = self.samples[idx]
        
        # 加载图片
        image = Image.open(data['img_path']).convert('RGB')
        
        return {
            'image': image,
            'text': data['raw_caption'],    # 用于送入模型计算loss
            'origin': data['origin_caption'], # 用于记录/对比
            'poisoned': data['poisoned']
        }

def collate_fn(batch, processor):
    """整理 batch 数据"""
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    
    # Processor 处理
    inputs = processor(
        text=texts, 
        images=images, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=77
    )
    
    # 附加其他信息
    inputs['poisoned_flags'] = torch.tensor([item['poisoned'] for item in batch])
    return inputs

In [ ]:
# ================= 配置区域 =================
CSV_PATH = "./clip/data/processed_dataset.csv" # 您的CSV路径
IMG_ROOT = "./clip/data"                       # 图片根目录

# CSV_PATH = "./clip/data_ca101/processed_dataset.csv" # 您的CSV路径
# IMG_ROOT = "./clip/data_ca101"         
MODEL_NAME = "openai/clip-vit-base-patch32"
CACHE_DIR = "./model/clip/clip_cache"
BATCH_SIZE = 8  # 根据显存调整

# ================= 初始化 Processor =================
processor = CLIPProcessor.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)

# ================= 实例化 Dataset =================
# 仅加载 train 集进行训练演示，如需测试集可改为 split='test'
train_dataset = BackdoorDataset(CSV_PATH, IMG_ROOT, split='train', processor=processor)

# ================= 实例化 DataLoader =================
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=lambda x: collate_fn(x, processor)
)

print(f"DataLoader 准备就绪，Batch 数量: {len(train_loader)}")

In [ ]:
# ==========================================
# 查看数据加载情况
# ==========================================

print("\n" + "="*30)
print("1. 检查 Dataset 展开结果 (查看前 5 个样本)")
print("="*30)

# 检查 train_dataset 的原始长度（展开前）
original_len = len(pd.read_csv(CSV_PATH).query("split == 'train'"))
print(f"原始 CSV 训练集行数: {original_len}")
print(f"Dataset 展开后样本数: {len(train_dataset)} (理论应为: {original_len * 5})")

# 打印前 5 个展开后的样本
for i in range(min(5, len(train_dataset))):
    sample = train_dataset[i]
    print(f"\n[样本 {i}]")
    print(f"  图片路径: {sample['image']}")
    print(f"  是否中毒: {'是' if sample['poisoned'] == 1 else '否'}")
    print(f"  Raw Caption : {sample['text'][:60]}...") # 只打印前60个字符
    print(f"  Origin Text : {sample['origin'][:60]}...")

print("\n" + "="*30)
print("2. 检查 DataLoader Batch 整理结果")
print("="*30)

# 获取一个 batch
try:
    # next(iter(train_loader)) 会返回一个 batch
    one_batch = next(iter(train_loader))
    
    # 打印 batch 中的键
    print(f"Batch 包含的键: {one_batch.keys()}")
    
    # 查看第一个样本在 batch 中的情况
    print(f"\nBatch 中第一张图片的 Tensor shape: {one_batch['pixel_values'][0].shape}")
    print(f"Batch 中第一个文本的 input_ids : {one_batch['input_ids'][0][:10]}... (前10个token)")
    
    # 解码文本以便查看
    decoded_text = processor.decode(one_batch['input_ids'][0], skip_special_tokens=True)
    print(f"Batch 中第一个文本解码结果: {decoded_text}")
    
    print("\n数据格式检查通过！")
    
except StopIteration:
    print("DataLoader 为空，无法获取数据。")
except Exception as e:
    print(f"发生错误: {e}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 加载原始 CLIP 模型
print("加载原始 CLIP 模型...")
model = CLIPModel.from_pretrained(
            MODEL_NAME,
            cache_dir=CACHE_DIR,
            use_safetensors=True
        )
# 2. 配置 LoRA
# target_modules 指定要微调的层，这里选择 Attention 的 Q, V 投影层
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.1,
    bias="none"
)

# 3. 应用 LoRA
model = get_peft_model(model, lora_config)
model.to(device)

# 打印可训练参数量
model.print_trainable_parameters()

In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("开始训练...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        # 移动数据到设备
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'poisoned_flags'}
        
        # 前向传播
        outputs = model(**inputs)
        
        # 计算 Loss
        # CLIP 的输出包含 logits_per_image 和 logits_per_text
        # 我们需要构建标签：对角线为正确配对 (第i张图对应第i段文本)
        logits_per_image = outputs.logits_per_image
        batch_size = logits_per_image.shape[0]
        ground_truth = torch.arange(batch_size, dtype=torch.long, device=device)
        
        # 图像->文本 Loss + 文本->图像 Loss
        loss_img = torch.nn.functional.cross_entropy(logits_per_image, ground_truth)
        loss_txt = torch.nn.functional.cross_entropy(outputs.logits_per_text, ground_truth)
        loss = (loss_img + loss_txt) / 2
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Finished. Average Loss: {avg_loss:.4f}")

print("训练完成！")

In [ ]:
# ================= 保存模型 =================
# 保存 LoRA 适配器权重 (非常小，通常只有几MB)
OUTPUT_DIR = "./clip/clip_lora"

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"模型 LoRA 权重已保存至: {OUTPUT_DIR}")
print("加载方式: model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)")

In [ ]:
import numpy as np
import torch
from tqdm import tqdm
import os
import pandas as pd

def evaluate_and_save(model, processor, dataset, output_csv_path, device):
    """
    测试函数：计算图片与 raw/origin 文本的相似度，并保存结果
    """
    model.eval()
    results = []
    
    print("开始测试集评估...")
    # 使用 DataLoader 批量处理
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=lambda x: x)
    
    with torch.no_grad():
        for batch_samples in tqdm(dataloader, desc="Evaluating"):
            # 准备数据
            images = [s['image'] for s in batch_samples]
            raw_texts = [s['text'] for s in batch_samples]
            origin_texts = [s['origin'] for s in batch_samples]
            poisoned_flags = [s['poisoned'] for s in batch_samples]
            
            # 1. 处理图片
            image_inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            # 修复点：使用 .pooler_output 获取特征向量
            img_out = model.get_image_features(**image_inputs)
            image_embeds = img_out.pooler_output 
            image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
            
            # 2. 处理 Raw (中毒/修改后) 文本
            text_inputs_raw = processor(text=raw_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            # 修复点：使用 .pooler_output 获取特征向量
            txt_out_raw = model.get_text_features(**text_inputs_raw)
            text_embeds_raw = txt_out_raw.pooler_output
            text_embeds_raw = text_embeds_raw / text_embeds_raw.norm(p=2, dim=-1, keepdim=True)
            
            # 3. 处理 Origin (原始) 文本
            text_inputs_origin = processor(text=origin_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            # 修复点：获取输出对象
            txt_out_origin = model.get_text_features(**text_inputs_origin)
            # 修复点：正确属性名是 pooler_output，而不是 pooler_origin
            text_embeds_origin = txt_out_origin.pooler_output
            text_embeds_origin = text_embeds_origin / text_embeds_origin.norm(p=2, dim=-1, keepdim=True)
            
            # 4. 计算余弦相似度 (点积)
            sim_raw = (image_embeds * text_embeds_raw).sum(dim=-1).cpu().numpy()
            sim_origin = (image_embeds * text_embeds_origin).sum(dim=-1).cpu().numpy()
            
            # 5. 记录结果
            for i in range(len(batch_samples)):
                # 获取文件名，处理可能的路径问题
                try:
                    # 尝试从 batch_samples 中获取 img_path
                    fname = dataset.samples[i]['img_path'].split('/')[-1]
                except:
                    fname = f"sample_{i}"

                results.append({
                    'filename': fname,
                    'poisoned': poisoned_flags[i],
                    'sim_with_raw': float(sim_raw[i]),
                    'sim_with_origin': float(sim_origin[i]),
                    'raw_text': raw_texts[i][:50] + "...",
                    'origin_text': origin_texts[i][:50] + "..."
                })

    # 保存结果
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"测试结果已保存至: {output_csv_path}")
    
    # ==========================================
    # 打印统计信息 (含 Raw > Origin 的比例)
    # ==========================================
    
    # --- 中毒样本统计 ---
    poison_res = df_results[df_results['poisoned'] == 1]
    if len(poison_res) > 0:
        raw_mean = poison_res['sim_with_raw'].mean()
        origin_mean = poison_res['sim_with_origin'].mean()
        
        # 计算Raw > Origin的样本数量和比例
        # 即：图片更倾向于与中毒文本匹配的情况
        higher_count = (poison_res['sim_with_raw'] >= poison_res['sim_with_origin']).sum()
        total_count = len(poison_res)
        ratio = (higher_count / total_count) * 100
        
        print(f"\n[中毒样本统计] (共 {total_count} 条)")
        print(f"  图片与Raw(中毒)文本平均相似度: {raw_mean:.4f}")
        print(f"  图片与Origin(原始)文本平均相似度: {origin_mean:.4f}")
        print(f"  --------------------------------------------------")
        print(f"  Raw > Origin 的样本数: {higher_count} / {total_count}")
        print(f"  Raw > Origin 的样本比例: {ratio:.2f}%")
        print(f"  (解释: 有 {ratio:.2f}% 的中毒图片更匹配中毒文本，此比例越高说明后门效果越好)")
    
    # --- 干净样本统计 ---
    clean_res = df_results[df_results['poisoned'] == 0]
    if len(clean_res) > 0:
        raw_mean_c = clean_res['sim_with_raw'].mean()
        origin_mean_c = clean_res['sim_with_origin'].mean()
        
        # 对于干净样本，Raw和Origin内容一样，理论上应该接近50%或相似度非常接近
        higher_count_c = (clean_res['sim_with_raw'] > clean_res['sim_with_origin']).sum()
        total_count_c = len(clean_res)
        ratio_c = (higher_count_c / total_count_c) * 100
        
        print(f"\n[干净样本统计] (共 {total_count_c} 条)")
        print(f"  图片与Raw文本平均相似度: {raw_mean_c:.4f}")
        print(f"  图片与Origin文本平均相似度: {origin_mean_c:.4f}")
        print(f"  Raw > Origin 的样本比例: {ratio_c:.2f}% (干净的图片匹配中毒文本的比例，理论上越低越好)")

    return df_results

# ==========================================
# 执行测试
# ==========================================
# 确保路径变量已定义
# CSV_PATH, IMG_ROOT, OUTPUT_DIR, model, processor 应在之前的 cell 中定义

test_dataset = BackdoorDataset(CSV_PATH, IMG_ROOT, split='test', processor=processor)
result_path = os.path.join("./clip", "evaluation_results.csv")

# 确保设备正确
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 运行评估
eval_results = evaluate_and_save(model, processor, test_dataset, result_path, DEVICE)

test_dataset = BackdoorDataset(CSV_PATH, IMG_ROOT, split='normal', processor=processor)
eval_results = evaluate_and_save(model, processor, test_dataset, result_path, DEVICE)